# Notebook 03: Wildfire Clustering Around Lake Tahoe

**Obstacle-Aware Clustering for Geographic Data**

This notebook applies the full obstacle-aware k-Means algorithm to real-world wildfire occurrence data around Lake Tahoe. Unlike Notebook 01 (spatial only), this analysis uses the complete dual-domain distance metric with geographic, arc-length, *and* attribute features — where all three weight parameters ($\alpha$, $\beta$, $\gamma$) come into play.

**Applied story**: Define fire management zones around the lake that are both spatially connected (fires in the same zone are reachable without going around the lake) and behaviorally similar (fires in the same zone share similar size and cause profiles).

---

## 1. Setup

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
from scipy.integrate import IntegrationWarning
import warnings
warnings.filterwarnings('ignore', category=IntegrationWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import sqlite3
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

# Our custom package
from obstacle_clustering import (
    SplineBoundary, ObstacleKMeans,
    optimize_weights, objective_surface, attribute_separation,
    plot_clusters, plot_comparison
)

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 2. Loading the Lake Tahoe Boundary

We load the cleaned boundary coordinates saved in Notebook 02 and fit the `SplineBoundary` for arc-length computation.

In [2]:
# Load boundary from Notebook 02
boundary_df = pd.read_csv('../data/boundaries/lake_tahoe_boundary.csv')
lon_boundary = boundary_df['longitude'].values
lat_boundary = boundary_df['latitude'].values

# Fit spline boundary
boundary = SplineBoundary(x_coords=lon_boundary, y_coords=lat_boundary)

L = boundary.total_arc_length()
print(f'Lake Tahoe boundary: {len(lon_boundary)} vertices')
print(f'Total arc length: {L:.4f} degrees')

# Sample for plotting
spline_pts = boundary.sample_boundary(n_points=500)

Lake Tahoe boundary: 269 vertices
Total arc length: 1.2027 degrees


## 3. Loading Wildfire Data

The [Fire Program Analysis Fire-Occurrence Database (FPA FOD)](https://www.fs.usda.gov/rds/archive/catalog/RDS-2013-0009.6) is a national database of wildfires reported in the United States from 1992 to 2020. We query the SQLite database for fires in the Lake Tahoe area.

**Important**: The FPA FOD SQLite file is not included in this repository due to its size (~2 GB). To reproduce this analysis:
1. Download `RDS-2013-0009.6_Data_Format4_SQLITE.zip` from the [USFS Research Data Archive](https://www.fs.usda.gov/rds/archive/catalog/RDS-2013-0009.6)
2. Rename to `fires.SQLITE` and place it in the `data/raw/` directory

In [3]:
# Connect to the FPA FOD SQLite database
db_path = '../data/raw/fires.sqlite'  # adjust path if needed
conn = sqlite3.connect(db_path)

# Query fires in the Lake Tahoe area
query = """
    SELECT LATITUDE, LONGITUDE, FIRE_SIZE, NWCG_GENERAL_CAUSE,
           NWCG_CAUSE_CLASSIFICATION, FIRE_YEAR, DISCOVERY_DOY
    FROM Fires
    WHERE LATITUDE BETWEEN 38.85 AND 39.35
      AND LONGITUDE BETWEEN -120.25 AND -119.85
"""

fires_raw = pd.read_sql_query(query, conn)
conn.close()

print(f'Total fires in Lake Tahoe area: {len(fires_raw)}')
print(f'Year range: {fires_raw["FIRE_YEAR"].min()} - {fires_raw["FIRE_YEAR"].max()}')
print(f'\nCause breakdown:')
print(fires_raw['NWCG_GENERAL_CAUSE'].value_counts().to_string())

Total fires in Lake Tahoe area: 1888
Year range: 1992 - 2020

Cause breakdown:
NWCG_GENERAL_CAUSE
Recreation and ceremony                       500
Missing data/not specified/undetermined       423
Natural                                       374
Smoking                                       181
Arson/incendiarism                             97
Misuse of fire by a minor                      87
Debris and open burning                        86
Equipment and vehicle use                      78
Power generation/transmission/distribution     44
Fireworks                                      11
Other causes                                    6
Railroad operations and maintenance             1


## 4. Data Cleaning and Feature Engineering

We prepare the data for clustering in several steps:

1. **Filter out "Missing data/not specified/undetermined" cause entries** — these don't contribute meaningful attribute information
2. **Encode cause as binary**: Natural (lightning) vs. Human (all other known causes) — this parallels the NWCG classification and creates a clean attribute for separation analysis
3. **Project each fire onto the lake boundary** to compute its arc-length parameter $s$
4. **Normalize features** so that geographic coordinates and fire size are on comparable scales with $s$

In [ ]:
# --- Step 1: Filter out missing cause data ---
fires = fires_raw[
    fires_raw['NWCG_GENERAL_CAUSE'] != 'Missing data/not specified/undetermined'
].copy()
print(f'Fires after filtering missing cause: {len(fires)}')

# --- Step 2: Encode cause as binary ---
# Natural = lightning; Human = all other known causes
fires['cause_binary'] = (
    fires['NWCG_GENERAL_CAUSE'].apply(lambda x: 0 if x == 'Natural' else 1)
)

print(f'\nCause classification:')
print(f"  Natural (lightning): {(fires['cause_binary'] == 0).sum()}")
print(f"  Human-caused:        {(fires['cause_binary'] == 1).sum()}")

# FIRE_SIZE is in acres (per FPA FOD documentation)
print(f'\nFire size statistics:')
print(fires['FIRE_SIZE'].describe())

Fires after filtering missing cause: 1465

Cause classification:
  Natural (lightning): 374
  Human-caused:        1091

Fire size statistics:
count    1465.000000
mean        9.381433
std       156.742888
min         0.010000
25%         0.100000
50%         0.100000
75%         0.100000
max      4222.000000
Name: FIRE_SIZE, dtype: float64


In [ ]:
# --- Step 3: Project each fire onto the lake boundary ---
print('Projecting fire locations onto lake boundary...')
t_values = []
s_values = []

for _, row in fires.iterrows():
    t_proj, s_proj = boundary.project_point(row['LONGITUDE'], row['LATITUDE'])
    t_values.append(t_proj)
    s_values.append(s_proj)

fires['t_param'] = t_values
fires['s_param'] = s_values

print(f'Projection complete: {len(fires)} fires projected')
print(f's range: [{fires["s_param"].min():.4f}, {fires["s_param"].max():.4f}]')

### Fire Size Distribution

Fire sizes in the FPA FOD span many orders of magnitude (from tiny spot fires to thousands of acres). We examine the distribution to decide on an appropriate transformation for clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
ax = axes[0]
ax.hist(fires['FIRE_SIZE'], bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel('Fire Size (acres)')
ax.set_ylabel('Count')
ax.set_title('Fire Size Distribution (Raw)')
ax.grid(True, alpha=0.3)

# Log-transformed
ax = axes[1]
ax.hist(np.log1p(fires['FIRE_SIZE']), bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel('log(1 + Fire Size)')
ax.set_ylabel('Count')
ax.set_title('Fire Size Distribution (Log-Transformed)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Use log-transformed fire size for clustering
fires['fire_size_log'] = np.log1p(fires['FIRE_SIZE'])

In [ ]:
# --- Step 4: Normalize features ---
# Geographic coordinates: Min-Max scale to [0, 1]
scaler_xy = MinMaxScaler()
xy_scaled = scaler_xy.fit_transform(fires[['LONGITUDE', 'LATITUDE']].values)

# Fire size (log): Min-Max scale to [0, 1]
scaler_size = MinMaxScaler()
fire_size_scaled = scaler_size.fit_transform(fires[['fire_size_log']].values)

# s is already in [0, 1]
# cause_binary is already 0 or 1

# Assemble feature matrix: [x_scaled, y_scaled, s, fire_size_scaled, cause_binary]
X = np.column_stack([
    xy_scaled,                          # columns 0, 1: geographic
    fires['s_param'].values,            # column 2: arc-length
    fire_size_scaled.ravel(),           # column 3: fire size (normalized)
    fires['cause_binary'].values        # column 4: cause type
])

feature_names = ['x_scaled', 'y_scaled', 's', 'fire_size', 'cause_type']
print(f'Feature matrix shape: {X.shape}')
print(f'Features: {feature_names}')
print(f'\nFirst 5 rows:')
print(np.array2string(X[:5], precision=4, suppress_small=True))

## 5. Visualizing the Raw Data

Before clustering, we examine the spatial distribution of fires around the lake, colored by cause type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# --- By cause ---
ax = axes[0]
ax.plot(spline_pts[:, 0], spline_pts[:, 1], 'k--', linewidth=1.5, alpha=0.7)
ax.fill(spline_pts[:, 0], spline_pts[:, 1], alpha=0.1, color='steelblue')

natural = fires[fires['cause_binary'] == 0]
human = fires[fires['cause_binary'] == 1]

ax.scatter(human['LONGITUDE'], human['LATITUDE'],
           c='#e74c3c', s=15, alpha=0.5, label=f'Human ({len(human)})', zorder=3)
ax.scatter(natural['LONGITUDE'], natural['LATITUDE'],
           c='#3498db', s=15, alpha=0.5, label=f'Natural ({len(natural)})', zorder=3)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Fires by Cause Type')
ax.legend(fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- By arc-length position ---
ax = axes[1]
scatter = ax.scatter(fires['LONGITUDE'], fires['LATITUDE'],
                     c=fires['s_param'], cmap='hsv', s=15, alpha=0.6, zorder=3)
ax.plot(spline_pts[:, 0], spline_pts[:, 1], 'k--', linewidth=1.5, alpha=0.7)
ax.fill(spline_pts[:, 0], spline_pts[:, 1], alpha=0.1, color='steelblue')
plt.colorbar(scatter, ax=ax, label='Arc-length $s$', shrink=0.6, aspect=20, pad=0.02)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Fires by Arc-Length Position')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.suptitle('Wildfire Distribution Around Lake Tahoe (1992–2020)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 6. Standard k-Means (Baseline)

As a baseline, we run standard k-Means using only the geographic coordinates $(x, y)$. This ignores the lake as an obstacle and ignores fire attributes entirely.

In [ ]:
# Standard k-Means on (x, y) only
k = 3
kmeans_standard = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_standard = kmeans_standard.fit_predict(xy_scaled)

print(f'Standard k-Means converged in {kmeans_standard.n_iter_} iterations')
print(f'\nCluster sizes:')
for i in range(k):
    n = np.sum(labels_standard == i)
    print(f'  Cluster {i+1}: {n} fires')

In [ ]:
# Plot standard k-Means results
fig, ax = plt.subplots(figsize=(10, 10))

colors = cm.get_cmap('tab10', k)
for i in range(k):
    mask = labels_standard == i
    ax.scatter(fires['LONGITUDE'].values[mask], fires['LATITUDE'].values[mask],
               c=[colors(i)], s=15, alpha=0.5, label=f'Cluster {i+1}', zorder=3)

ax.plot(spline_pts[:, 0], spline_pts[:, 1], 'k--', linewidth=1.5, alpha=0.7)
ax.fill(spline_pts[:, 0], spline_pts[:, 1], alpha=0.1, color='steelblue')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Standard k-Means (Geographic Only)')
ax.legend(fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check attribute separation for baseline
sigma_std, details_std = attribute_separation(
    X, labels_standard, k, attr_indices=[3, 4]
)
print(f'\nBaseline attribute separation (sigma_a): {sigma_std:.4f}')

## 7. Obstacle-Aware k-Means (Fixed Weights)

Before optimizing, we run the obstacle-aware algorithm with equal weights to see the effect of including the arc-length parameter and attributes:

$$d^2 = \alpha^2 \cdot \text{(geography)} + \beta^2 \cdot \text{(arc-length)} + \gamma^2 \cdot \text{(attributes)}$$

We use $\alpha = 1.0$, $\beta = 1.0$, $\gamma = 1.0$ as a starting point.

In [ ]:
# Obstacle-aware k-Means with equal weights
model_equal = ObstacleKMeans(
    k=k,
    boundary=boundary,
    alpha=1.0,
    beta=1.0,
    gamma=1.0,
    random_state=42,
    n_attr=2             # two attribute features: fire_size, cause_type
)
model_equal.fit(X, t_data=fires['t_param'].values)

labels_equal = model_equal.labels_

print(f'Obstacle-aware k-Means (equal weights) converged in {model_equal.n_iter_} iterations')
print(f'rho_bar (geographic coherence): {model_equal.rho_bar_:.4f}')

print(f'\nCluster sizes:')
for i in range(k):
    n = np.sum(labels_equal == i)
    print(f'  Cluster {i+1}: {n} fires')

# Attribute separation
sigma_eq, details_eq = attribute_separation(X, labels_equal, k, attr_indices=[3, 4])
print(f'\nAttribute separation (sigma_a): {sigma_eq:.4f}')
print(f'Objective J = rho_bar + (1 - sigma_a): {model_equal.rho_bar_ + (1 - sigma_eq):.4f}')

In [ ]:
# Plot obstacle-aware results (equal weights)
fig, ax = plt.subplots(figsize=(10, 10))

for i in range(k):
    mask = labels_equal == i
    ax.scatter(fires['LONGITUDE'].values[mask], fires['LATITUDE'].values[mask],
               c=[colors(i)], s=15, alpha=0.5, label=f'Cluster {i+1}', zorder=3)

ax.plot(spline_pts[:, 0], spline_pts[:, 1], 'k--', linewidth=1.5, alpha=0.7)
ax.fill(spline_pts[:, 0], spline_pts[:, 1], alpha=0.1, color='steelblue')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Obstacle-Aware k-Means ($\alpha=1, \beta=1, \gamma=1$)')
ax.legend(fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Cluster Profiles

To understand what the algorithm is finding, we examine the attribute composition of each cluster — the distribution of fire sizes and the proportion of natural vs. human-caused fires.

In [ ]:
# Add cluster labels to the dataframe for analysis
fires['cluster_equal'] = labels_equal

fig, axes = plt.subplots(k, 2, figsize=(14, 4 * k))

for i in range(k):
    cluster = fires[fires['cluster_equal'] == i]

    # Fire size distribution
    ax = axes[i, 0]
    ax.hist(cluster['fire_size_log'], bins=30, color=colors(i),
            edgecolor='white', alpha=0.7)
    ax.set_xlabel('log(1 + Fire Size)')
    ax.set_ylabel('Count')
    ax.set_title(f'Cluster {i+1}: Fire Size (n={len(cluster)})')
    ax.grid(True, alpha=0.3)

    # Cause breakdown
    ax = axes[i, 1]
    cause_counts = cluster['cause_binary'].value_counts().sort_index()
    labels_cause = ['Natural', 'Human']
    bar_colors = ['#3498db', '#e74c3c']
    bars = ax.bar(labels_cause, [cause_counts.get(0, 0), cause_counts.get(1, 0)],
                  color=bar_colors, edgecolor='white')
    ax.set_ylabel('Count')
    ax.set_title(f'Cluster {i+1}: Cause Type')
    ax.grid(True, alpha=0.3)

plt.suptitle('Cluster Attribute Profiles (Equal Weights)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# Summary table
print('Cluster summary:')
print(f'{"Cluster":<10} {"Size":<8} {"Mean Fire Size":<16} {"% Natural":<12} {"Mean s":<10}')
print('-' * 56)
for i in range(k):
    cluster = fires[fires['cluster_equal'] == i]
    print(f'{i+1:<10} {len(cluster):<8} '
          f'{cluster["FIRE_SIZE"].mean():<16.2f} '
          f'{(cluster["cause_binary"] == 0).mean() * 100:<12.1f} '
          f'{cluster["s_param"].mean():<10.3f}')

## 9. Hyperparameter Optimization

The weights $\alpha$, $\beta$, and $\gamma$ control the balance between geographic coherence and attribute separation. The objective function

$$J = \bar{\rho} + (1 - \sigma_a)$$

is minimized by simulated annealing (`dual_annealing` from SciPy). Lower $J$ means tighter spatial clusters *and* better attribute differentiation between clusters.

**Note**: This cell may take several minutes to run, as it fits the k-Means model hundreds of times during the annealing process.

In [ ]:
# Optimize weights via simulated annealing
print('Running simulated annealing optimization...')
print('(This may take several minutes)')

result = optimize_weights(
    X=X,
    t_data=fires['t_param'].values,
    boundary=boundary,
    k=k,
    n_attr=2,
    attr_indices=[3, 4],        # fire_size and cause_type columns
    bounds=[(0, 2), (0, 2), (0, 2)],
    random_state=42,
    maxiter=200                  # reduce if too slow; increase for finer search
)

print(f'\nOptimization complete!')
print(f'  Optimal alpha: {result["alpha"]:.4f}')
print(f'  Optimal beta:  {result["beta"]:.4f}')
print(f'  Optimal gamma: {result["gamma"]:.4f}')
print(f'  Objective J:   {result["objective"]:.4f}')
print(f'  rho_bar:       {result["rho_bar"]:.4f}')
print(f'  sigma_a:       {result["sigma_a"]:.4f}')

In [ ]:
# Extract the optimized model
model_opt = result['model']
labels_opt = model_opt.labels_

print('Optimized cluster sizes:')
for i in range(k):
    n = np.sum(labels_opt == i)
    print(f'  Cluster {i+1}: {n} fires')

## 10. Objective Surface

To understand the optimization landscape, we evaluate the objective function over a grid of $(\alpha, \beta)$ values with $\gamma$ fixed at the optimal value. This produces a contour plot showing how the objective varies with the balance between geographic and arc-length weights.

In [ ]:
# Compute objective surface
print('Computing objective surface...')
print('(This may take several minutes)')

alpha_range = np.linspace(0.1, 2.0, 15)
beta_range = np.linspace(0.1, 2.0, 15)

alpha_grid, beta_grid, obj_grid = objective_surface(
    X=X,
    t_data=fires['t_param'].values,
    boundary=boundary,
    alpha_range=alpha_range,
    beta_range=beta_range,
    gamma=result['gamma'],      # fix gamma at optimal value
    k=k,
    n_attr=2,
    attr_indices=[3, 4],
    random_state=42
)

print('Surface computation complete.')

In [ ]:
# Plot the objective surface
fig, ax = plt.subplots(figsize=(10, 8))

contour = ax.contourf(alpha_grid, beta_grid, obj_grid, levels=30, cmap='viridis')
plt.colorbar(contour, ax=ax, label='Objective $J = \\bar{\\rho} + (1 - \\sigma_a)$')

# Mark the optimum
ax.scatter(result['alpha'], result['beta'],
           c='red', marker='*', s=300, edgecolors='white',
           linewidth=1.5, zorder=5, label='Optimum')

ax.set_xlabel('$\\alpha$ (geographic weight)', fontsize=12)
ax.set_ylabel('$\\beta$ (arc-length weight)', fontsize=12)
ax.set_title(f'Objective Surface ($\\gamma$ = {result["gamma"]:.2f})', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 11. Optimized Results

We compare the standard k-Means, equal-weight obstacle-aware, and optimized obstacle-aware results side by side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

titles = [
    'Standard k-Means\n(geographic only)',
    'Obstacle-Aware\n($\\alpha=1, \\beta=1, \\gamma=1$)',
    f'Obstacle-Aware (Optimized)\n($\\alpha$={result["alpha"]:.2f}, '
    f'$\\beta$={result["beta"]:.2f}, $\\gamma$={result["gamma"]:.2f})'
]
all_labels = [labels_standard, labels_equal, labels_opt]

for col, (labels, title) in enumerate(zip(all_labels, titles)):
    ax = axes[col]
    for i in range(k):
        mask = labels == i
        ax.scatter(fires['LONGITUDE'].values[mask], fires['LATITUDE'].values[mask],
                   c=[colors(i)], s=12, alpha=0.5, label=f'Cluster {i+1}', zorder=3)

    ax.plot(spline_pts[:, 0], spline_pts[:, 1], 'k--', linewidth=1.5, alpha=0.7)
    ax.fill(spline_pts[:, 0], spline_pts[:, 1], alpha=0.1, color='steelblue')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparison: Standard vs. Obstacle-Aware Clustering', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Optimized cluster profiles
fires['cluster_opt'] = labels_opt

fig, axes = plt.subplots(k, 2, figsize=(14, 4 * k))

for i in range(k):
    cluster = fires[fires['cluster_opt'] == i]

    # Fire size distribution
    ax = axes[i, 0]
    ax.hist(cluster['fire_size_log'], bins=30, color=colors(i),
            edgecolor='white', alpha=0.7)
    ax.set_xlabel('log(1 + Fire Size)')
    ax.set_ylabel('Count')
    ax.set_title(f'Cluster {i+1}: Fire Size (n={len(cluster)})')
    ax.grid(True, alpha=0.3)

    # Cause breakdown
    ax = axes[i, 1]
    cause_counts = cluster['cause_binary'].value_counts().sort_index()
    labels_cause = ['Natural', 'Human']
    bar_colors = ['#3498db', '#e74c3c']
    bars = ax.bar(labels_cause, [cause_counts.get(0, 0), cause_counts.get(1, 0)],
                  color=bar_colors, edgecolor='white')
    ax.set_ylabel('Count')
    ax.set_title(f'Cluster {i+1}: Cause Type')
    ax.grid(True, alpha=0.3)

plt.suptitle('Cluster Attribute Profiles (Optimized Weights)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# Summary table
print('\nOptimized cluster summary:')
print(f'{"Cluster":<10} {"Size":<8} {"Mean Fire Size":<16} {"% Natural":<12} {"Mean s":<10}')
print('-' * 56)
for i in range(k):
    cluster = fires[fires['cluster_opt'] == i]
    print(f'{i+1:<10} {len(cluster):<8} '
          f'{cluster["FIRE_SIZE"].mean():<16.2f} '
          f'{(cluster["cause_binary"] == 0).mean() * 100:<12.1f} '
          f'{cluster["s_param"].mean():<10.3f}')

## 12. Quantitative Comparison

We compare the three approaches quantitatively: geographic coherence ($\bar{\rho}$), attribute separation ($\sigma_a$), and the composite objective ($J$).

In [ ]:
# Compute metrics for all three approaches
sigma_std, _ = attribute_separation(X, labels_standard, k, attr_indices=[3, 4])
sigma_eq, _ = attribute_separation(X, labels_equal, k, attr_indices=[3, 4])
sigma_opt, _ = attribute_separation(X, labels_opt, k, attr_indices=[3, 4])

# For rho_bar, refit standard k-Means as ObstacleKMeans with alpha=1, beta=0, gamma=0
# to get a comparable rho_bar measure
model_std_refit = ObstacleKMeans(
    k=k, boundary=boundary, alpha=1.0, beta=0.0, gamma=0.0,
    random_state=42, n_attr=2
)
model_std_refit.fit(X, t_data=fires['t_param'].values)

print('Quantitative comparison:')
print(f'{"Method":<30} {"rho_bar":>10} {"sigma_a":>10} {"J":>10}')
print('-' * 62)

rho_std = model_std_refit.rho_bar_
j_std = rho_std + (1 - sigma_std)
print(f'{"Standard k-Means":<30} {rho_std:>10.4f} {sigma_std:>10.4f} {j_std:>10.4f}')

rho_eq = model_equal.rho_bar_
j_eq = rho_eq + (1 - sigma_eq)
print(f'{"Obstacle-Aware (equal)":<30} {rho_eq:>10.4f} {sigma_eq:>10.4f} {j_eq:>10.4f}')

rho_opt = model_opt.rho_bar_
j_opt = rho_opt + (1 - sigma_opt)
print(f'{"Obstacle-Aware (optimized)":<30} {rho_opt:>10.4f} {sigma_opt:>10.4f} {j_opt:>10.4f}')

## 13. Saving Results

We save the clustered fire data and optimal parameters for use in Notebook 04 (results and interactive maps).

In [ ]:
# Save clustered fire data
output_cols = ['LATITUDE', 'LONGITUDE', 'FIRE_SIZE', 'NWCG_GENERAL_CAUSE',
               'cause_binary', 'FIRE_YEAR', 'DISCOVERY_DOY',
               't_param', 's_param', 'fire_size_log',
               'cluster_equal', 'cluster_opt']
fires[output_cols].to_csv('../data/processed/tahoe_fires_clustered.csv', index=False)

# Save optimal parameters
import json
opt_params = {
    'alpha': result['alpha'],
    'beta': result['beta'],
    'gamma': result['gamma'],
    'k': k,
    'objective': result['objective'],
    'rho_bar': result['rho_bar'],
    'sigma_a': result['sigma_a'],
    'n_fires': len(fires),
}
with open('../data/processed/optimal_params.json', 'w') as f:
    json.dump(opt_params, f, indent=2)

print(f'Saved {len(fires)} clustered fires to data/processed/tahoe_fires_clustered.csv')
print(f'Saved optimal parameters to data/processed/optimal_params.json')

## 14. Discussion

This notebook demonstrated the full obstacle-aware clustering workflow on real-world wildfire data:

1. **Data pipeline**: We loaded wildfire occurrence records from the FPA FOD database, projected each fire onto the Lake Tahoe boundary to compute its arc-length parameter $s$, and engineered a feature vector combining geography, boundary position, and fire attributes.

2. **Dual-domain clustering**: Unlike Notebook 01 (spatial only, $\gamma = 0$), this analysis used the complete weighted distance metric with all three weight parameters. This allows the algorithm to find clusters that are simultaneously spatially coherent around the obstacle and differentiated by their attribute profiles.

3. **Hyperparameter optimization**: Simulated annealing found the weight combination that best balances geographic coherence ($\bar{\rho}$) and attribute separation ($\sigma_a$). The objective surface visualization shows how the optimization landscape varies with $\alpha$ and $\beta$.

4. **Comparison**: The side-by-side comparison of standard k-Means vs. obstacle-aware clustering illustrates the practical benefit of incorporating the arc-length parameter and fire attributes.

### Methodological Notes

- **Boundary choice**: We use the lake shoreline as the obstacle boundary because it defines the fundamental geographic barrier. The road network approximately follows the shoreline and would produce a similar arc-length ordering — see Notebook 02 for discussion.

- **Fire size transformation**: We use $\log(1 + \text{fire size})$ rather than raw acres because fire sizes span several orders of magnitude. The log transformation reduces the influence of rare large fires and puts the distribution on a more comparable scale with the other features.

- **Cause encoding**: The binary Natural/Human classification simplifies the NWCG cause categories into a single feature that captures the most important management distinction.

### Key Files Produced

| File | Description |
|------|-------------|
| `data/processed/tahoe_fires_clustered.csv` | Fire records with cluster labels and computed features |
| `data/processed/optimal_params.json` | Optimal $\alpha$, $\beta$, $\gamma$ and performance metrics |